# Day 3: ScannerAgent and MessagingAgent

## Order

1. Modal.com and SpecialistAgent  
2. RAG, FrontierAgent, EnsembleAgent  
3. ScannerAgent, MessagingAgent  
4. AutonomousPlanningAgent and DealAgentFramework  
5. Gradio finale

Today's piece: a ScannerAgent that watches a real wine shop for bargains.
The shop publishes a paginated public `products.json` (permitted by its
robots.txt), and about a third of the listings include a **printed critic
score** (Wine Advocate, James Suckling, Wine Spectator, Vinous) plus a quoted
tasting note. The scanner selects only those — a wine without an explicit
printed score is never surfaced — and extracts the structured fields the
rest of the system needs. Then a MessagingAgent pushes alerts to your phone.

In [ ]:
import logging
import os

import requests
from dotenv import load_dotenv
from openai import OpenAI

from utils.listings import Listing, ListingSelection, Opportunity, ScrapedListing

load_dotenv(override=True)
openai_client = OpenAI()
MODEL = "gpt-5-mini"

In [ ]:
listings = ScrapedListing.fetch(show_progress=True)
len(listings)

In [ ]:
scored = [listing for listing in listings if listing.has_score()]
print(f"{len(scored)} of {len(listings)} in-stock listings have a printed critic score")
print()
print(scored[0].describe())

### Asking gpt-5-mini to select and extract the 5 best-documented wines

Structured outputs guarantee the reply parses into our `ListingSelection`
model. The system prompt enforces the critic-only policy: no printed score,
no selection.

In [ ]:
from agents.scanner_agent import SYSTEM_PROMPT, ScannerAgent

print(SYSTEM_PROMPT)

In [ ]:
user_prompt = ScannerAgent.make_user_prompt(scored[:50])
print(user_prompt[:2000])

In [ ]:
response = openai_client.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ListingSelection,
    reasoning_effort="minimal",
)
results = response.choices[0].message.parsed
results

In [ ]:
for wine in results.listings:
    print(f"{wine.title} — {wine.points} pts at ${wine.price:.2f}")
    print(f"  {wine.variety} | {wine.region}, {wine.province}, {wine.country} | {wine.winery}")
    print(f"  {wine.tasting_note[:120]}...")
    print(f"  {wine.url}")
    print()

### The same flow as an agent

`ScannerAgent.scan(memory)` also deduplicates against previously surfaced
URLs, so repeat runs only ever return new wines.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
agent = ScannerAgent()
result = agent.scan()
result

# Optional: Pushover push notifications

Pushover sends push notifications to your phone. **This is optional** —
without keys, the MessagingAgent logs alerts instead of pushing, and
everything downstream works the same. To enable it later:

1. Sign up free at https://pushover.net/
2. Your **user key** is on the top right of the home screen (starts with `u`)
3. Click "Create an Application/API Token", any name, and copy its **token**
   (starts with `a`)
4. Add both to your `.env` as `PUSHOVER_USER` / `PUSHOVER_TOKEN`
5. Install the Pushover app on your phone and log in

In [ ]:
load_dotenv(override=True)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")

print(f"User key {'found, starts with ' + pushover_user[0] if pushover_user else 'NOT found'}")
print(f"App token {'found, starts with ' + pushover_token[0] if pushover_token else 'NOT found'}")

In [ ]:
def push(message: str) -> None:
    if not (pushover_user and pushover_token):
        print(f"(no Pushover keys — would push): {message}")
        return
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post("https://api.pushover.net/1/messages.json", data=payload, timeout=10)

In [ ]:
push("MASSIVE WINE BARGAIN!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE WINE BARGAIN!!")

### A structured alert

`alert()` takes an `Opportunity` — the record the planning agent will build
tomorrow: the scanned wine plus its actual VFM at the shop price versus the
ensemble's text-based estimate.

In [ ]:
sample = Opportunity(
    listing=Listing(
        title="2023 Platt Vineyard West Sonoma Coast Pinot Noir",
        tasting_note="Full-bodied yet lifted, with aromas of ripe red fruit.",
        points=95,
        price=82.99,
        url="https://bottlebarn.com/products/example",
        variety="Pinot Noir",
        country="US",
        province="California",
        region="Sonoma Coast",
        winery="Platt Vineyard",
    ),
    estimated_vfm=48,
    actual_vfm=61,
    delta=13,
)

agent.alert(sample)